# Plotly Dashboard Preparation

This notebook contains all interactive Plotly visualizations
used in the Streamlit dashboard for the UPI Impulse Analysis project.

Goals:
- Create interactive visualizations
- Improve storytelling
- Build dashboard-ready figures
- Replace static matplotlib charts

In [14]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

from plotly.subplots import make_subplots

In [15]:
df = pd.read_csv("../data/processed/combined_with_clusters.csv")

df.head()

,respondent_id,age_group,gender,college_year,monthly_budget_range,avg_monthly_budget,income_source,primary_upi_app,perceives_upi_risky,upi_usage_reason,...,cat_offline_cafe,cat_other,post_regret_action,hidden_purchase,regret_intensity,high_regret,regret_description,impulse_composite_score,cluster,cluster_name
0,1000,22-25,Male,Ug,Rs6000+,18000.0,100% Parents Money,PhonePe,1,Comfort,...,0,0,Accept it or move on,0,3,0,Marrow course,1.375,2.0,Controlled Spenders
1,1001,18-21,Male,Ug,Rs1000 - Rs3000,3500.0,100% Parents Money,PhonePe,1,Comfort,...,0,0,Accept it or move on,0,3,0,NaN,1.250,0.0,Routine Evening Spenders
2,1002,18-21,Male,Ug,Rs1000 - Rs3000,2000.0,100% Parents Money,PhonePe,1,Comfort,...,0,0,Accept it or move on,1,1,0,NaN,1.000,2.0,Controlled Spenders
3,1003,18-21,Male,Ug,Rs3000 - Rs6000,4500.0,100% Parents Money,Google Pay (GPay),1,Comfort,...,0,0,Accept it or move on,1,1,0,NaN,1.500,1.0,High Impulsive Spenders
4,1004,Under 18,Female,Ug,Rs3000 - Rs6000,4500.0,100% Parents Money,PhonePe,1,Comfort,...,1,0,Accept it or move on,0,2,0,NaN,1.375,0.0,Routine Evening Spenders


In [16]:
print("Shape:", df.shape)

print("\nColumns:\n")
print(df.columns.tolist())

Shape: (105, 47)

Columns:

['respondent_id', 'age_group', 'gender', 'college_year', 'monthly_budget_range', 'avg_monthly_budget', 'income_source', 'primary_upi_app', 'perceives_upi_risky', 'upi_usage_reason', 'weekly_tx_range', 'avg_weekly_tx', 'pct_unplanned', 'pct_unplanned_avg', 'balance_check_habit', 'tracks_expenses', 'ran_out_of_money', 'flag_morning', 'flag_afternoon', 'flag_evening', 'flag_latenight', 'flag_postmidnight', 'trigger_boredom', 'trigger_fomo', 'trigger_latenight', 'trigger_cashback', 'trigger_stress_relief', 'trigger_scarcity_notif', 'trigger_cart_abandon', 'trigger_exam_season', 'regret_frequency', 'cat_food_delivery', 'cat_grocery', 'cat_online_shopping', 'cat_subscriptions', 'cat_gaming', 'cat_gadgets', 'cat_offline_cafe', 'cat_other', 'post_regret_action', 'hidden_purchase', 'regret_intensity', 'high_regret', 'regret_description', 'impulse_composite_score', 'cluster', 'cluster_name']


In [17]:
plot_bg = "white"

common_layout = dict(
    template="plotly_white",
    title_x=0.5,
    font=dict(size=14),
    margin=dict(l=40, r=40, t=60, b=40)
)

In [18]:
total_users = len(df)

high_regret_pct = round(df["high_regret"].mean() * 100, 1)

avg_impulse_score = round(
    df["impulse_composite_score"].mean(),
    2
)

late_night_pct = round(
    df["flag_latenight"].mean() * 100,
    1
)

print("Total Users:", total_users)
print("High Regret %:", high_regret_pct)
print("Avg Impulse Score:", avg_impulse_score)
print("Late Night Buyers %:", late_night_pct)

Total Users: 105
High Regret %: 33.3
Avg Impulse Score: 1.74
Late Night Buyers %: 40.0


## User Demographics
### Gender Distribution

In [19]:
gender_counts = (
    df["gender"]
    .value_counts()
    .reset_index()
)

gender_counts.columns = ["gender", "count"]

fig = px.pie(
    gender_counts,
    names="gender",
    values="count",
    hole=0.55,
    title="Gender Distribution of Respondents",
)

fig.update_traces(
    textposition='inside',
    textinfo='percent+label'
)

fig.update_layout(
    **common_layout,
    showlegend=False
)

fig.show()

### College Year Distribution

In [20]:
year_counts = (
    df["college_year"]
    .value_counts()
    .reset_index()
)

year_counts.columns = ["college_year", "count"]

fig = px.bar(
    year_counts,
    x="college_year",
    y="count",
    text_auto=True,
    title="Undergraduate vs Postgraduate Respondents",
)

fig.update_layout(
    **common_layout,
    xaxis_title="College Year",
    yaxis_title="Number of Students"
)

fig.show()

# Spending Behaviour
## Impulse Purchase Timing

In [21]:
time_cols = {
    "flag_morning": "Morning",
    "flag_afternoon": "Afternoon",
    "flag_evening": "Evening",
    "flag_latenight": "Late Night",
    "flag_postmidnight": "Post Midnight"
}

time_data = pd.DataFrame({
    "Time": list(time_cols.values()),
    "Count": [
        df[col].sum()
        for col in time_cols.keys()
    ]
})

fig = px.bar(
    time_data,
    x="Count",
    y="Time",
    orientation="h",
    text_auto=True,
    title="When Do Students Make Impulsive Purchases?"
)

fig.update_layout(
    **common_layout,
    xaxis_title="Number of Respondents",
    yaxis_title=""
)

fig.show()

## Most Regretted Spending Categories

In [22]:
cat_cols = {
    "cat_food_delivery": "Food Delivery",
    "cat_grocery": "Quick Commerce",
    "cat_online_shopping": "Online Shopping",
    "cat_subscriptions": "Subscriptions",
    "cat_gaming": "Gaming",
    "cat_gadgets": "Gadgets",
    "cat_offline_cafe": "Offline Cafe",
    "cat_other": "Other"
}

cat_data = pd.DataFrame({
    "Category": list(cat_cols.values()),
    "Count": [
        df[col].sum()
        for col in cat_cols.keys()
    ]
})

cat_data = cat_data.sort_values(
    by="Count",
    ascending=True
)

fig = px.bar(
    cat_data,
    x="Count",
    y="Category",
    orientation="h",
    text_auto=True,
    title="Which Purchases Do Students Regret Most?"
)

fig.update_layout(
    **common_layout,
    xaxis_title="Number of Respondents",
    yaxis_title=""
)

fig.show()